1: INSTALACIÓN DE LIBRERÍAS

In [4]:
# Se ejecuta solo al inicio del proyecto.
!pip install -q pandas
!pip install -q langchain
!pip install -q langchain-mistralai
!pip install -q chromadb
!pip install -q tqdm
!pip install -q mistralai
!pip install -q numpy

print("✅ Todas las librerías se han instalado correctamente")

✅ Todas las librerías se han instalado correctamente


2: CONFIGURACIÓN DE API KEY DE MISTRAL

In [6]:
import os
from google.colab import userdata

# Obtener la API Key desde los Secrets de Colab
MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')

# Verificar que existe
if MISTRAL_API_KEY is None:
    raise ValueError("""
    ❌ ERROR: No se encontró la API Key de Mistral.

    SOLUCIÓN:
    1. Haz clic en el ícono de la llave 🔑 en el panel izquierdo de Colab.
    2. Agrega un nuevo secret con nombre: MISTRAL_API_KEY
    3. Pega tu clave de API (que obtuviste de console.mistral.ai)
    4. Vuelve a ejecutar esta celda.
    """)
else:
    os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY
    print("✅ API Key de Mistral configurada correctamente")
    print(f"   (Key: {MISTRAL_API_KEY[:8]}...{MISTRAL_API_KEY[-4:]})")

✅ API Key de Mistral configurada correctamente
   (Key: ZHAWKXoF...W9wq)


3: CARGA DEL DATASET DESDE PC LOCAL

In [8]:
import pandas as pd

# Mostrar todas las columnas
pd.set_option('display.max_columns', None)

# Cargar archivo ya subido
df = pd.read_csv("sales_data_sample.csv", encoding="latin1")

# Información básica
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

print("\nNombres de columnas:")
print(df.columns.tolist())

print("\nPrimeras 10 filas:")
display(df.head(10))

Filas: 2823
Columnas: 25

Nombres de columnas:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Primeras 10 filas:


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,PRODUCTLINE,MSRP,PRODUCTCODE,CUSTOMERNAME,PHONE,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,Motorcycles,95,S10_1678,Land of Toys Inc.,2125557818,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,Motorcycles,95,S10_1678,Toys4GrownUps.com,6265557265,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,Motorcycles,95,S10_1678,Corporate Gift Ideas Co.,6505551386,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium
5,10168,36,96.66,1,3479.76,10/28/2003 0:00,Shipped,4,10,2003,Motorcycles,95,S10_1678,Technics Stores Inc.,6505556809,9408 Furth Circle,NaN,Burlingame,CA,94217,USA,NaN,Hirano,Juri,Medium
6,10180,29,86.13,9,2497.77,11/11/2003 0:00,Shipped,4,11,2003,Motorcycles,95,S10_1678,Daedalus Designs Imports,20.16.1555,"184, chausse de Tournai",NaN,Lille,NaN,59000,France,EMEA,Rance,Martine,Small
7,10188,48,100.00,1,5512.32,11/18/2003 0:00,Shipped,4,11,2003,Motorcycles,95,S10_1678,Herkku Gifts,+47 2267 3215,"Drammen 121, PR 744 Sentrum",NaN,Bergen,NaN,N 5804,Norway,EMEA,Oeztan,Veysel,Medium
8,10201,22,98.57,2,2168.54,12/1/2003 0:00,Shipped,4,12,2003,Motorcycles,95,S10_1678,Mini Wheels Co.,6505555787,5557 North Pendale Street,NaN,San Francisco,CA,NaN,USA,NaN,Murphy,Julie,Small
9,10211,41,100.00,14,4708.44,1/15/2004 0:00,Shipped,1,1,2004,Motorcycles,95,S10_1678,Auto Canal Petit,(1) 47.55.6555,"25, rue Lauriston",NaN,Paris,NaN,75016,France,EMEA,Perrier,Dominique,Medium


4: LIMPIEZA Y NORMALIZACIÓN DE DATOS

In [11]:
import pandas as pd
import numpy as np

# ==========================================================
# VALIDACIÓN INICIAL
# ==========================================================
if 'df' not in globals():
    raise ValueError("❌ El DataFrame 'df' no existe. Carga el dataset primero.")

df_clean = df.copy()

print("=" * 70)
print("INICIO DEL PROCESO DE NORMALIZACIÓN Y LIMPIEZA")
print("=" * 70)

filas_iniciales = len(df_clean)

# ==========================================================
# PASO 1: INSPECCIÓN INICIAL
# ==========================================================
print("\n📌 PASO 1: Inspección inicial")
print(f"   Filas: {df_clean.shape[0]}")
print(f"   Columnas: {df_clean.shape[1]}")

# ==========================================================
# PASO 2: ESTANDARIZAR NOMBRES DE COLUMNAS
# ==========================================================
print("\n📌 PASO 2: Estandarización de nombres de columnas")
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.upper()
    .str.replace(" ", "_")
)
print("   ✅ Columnas normalizadas")

# ==========================================================
# PASO 2.5: ELIMINAR COLUMNAS INÚTILES
# ==========================================================
print("\n📌 PASO 2.5: Eliminación de columnas inútiles")

columnas_a_eliminar = [
    'ADDRESSLINE1', 'ADDRESSLINE2', 'PHONE',
    'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'POSTALCODE'
]

columnas_a_eliminar_existentes = [col for col in columnas_a_eliminar if col in df_clean.columns]

if columnas_a_eliminar_existentes:
    df_clean.drop(columns=columnas_a_eliminar_existentes, inplace=True)
    print(f"   ✅ Eliminadas: {', '.join(columnas_a_eliminar_existentes)}")

# ==========================================================
# PASO 3: ELIMINAR DUPLICADOS
# ==========================================================
print("\n📌 PASO 3: Eliminación de filas duplicadas")
duplicados_antes = df_clean.duplicated().sum()
df_clean.drop_duplicates(inplace=True)
print(f"   ✅ Duplicados eliminados: {duplicados_antes} → {df_clean.duplicated().sum()}")

# ==========================================================
# PASO 4: TRATAMIENTO DE NULOS
# ==========================================================
print("\n📌 PASO 4: Tratamiento de valores nulos")

# 4.1: Verificar nulos antes de cualquier conversión
print("\n   🔍 Análisis inicial de nulos:")
nulos_iniciales = df_clean.isnull().sum()
nulos_con_datos = nulos_iniciales[nulos_iniciales > 0]
for col, cantidad in nulos_con_datos.items():
    porcentaje = (cantidad / len(df_clean)) * 100
    print(f"      • {col}: {cantidad} nulos ({porcentaje:.1f}%)")

# 4.2: Columnas CRÍTICAS - Eliminar filas con datos faltantes
print("\n   🔧 Tratamiento de nulos:")
columnas_criticas = ['SALES', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERDATE']
columnas_criticas_existentes = [col for col in columnas_criticas if col in df_clean.columns]

antes_eliminacion = len(df_clean)
df_clean.dropna(subset=columnas_criticas_existentes, inplace=True)
eliminadas_criticas = antes_eliminacion - len(df_clean)
print(f"      ✅ Eliminadas por datos críticos faltantes: {eliminadas_criticas}")

# 4.3: Columnas de TEXTO - Reemplazar nulos por 'DESCONOCIDO'
columnas_texto = ['COUNTRY', 'PRODUCTLINE', 'CUSTOMERNAME', 'STATUS', 'TERRITORY', 'CITY', 'STATE']
for col in columnas_texto:
    if col in df_clean.columns:
        cantidad_nulos = df_clean[col].isnull().sum()
        if cantidad_nulos > 0:
            # IMPORTANTE: Convertir a string primero para poder asignar 'DESCONOCIDO'
            df_clean[col] = df_clean[col].astype(object).fillna('DESCONOCIDO')
            print(f"      ✅ {col}: {cantidad_nulos} nulos → 'DESCONOCIDO'")

# 4.4: Columna MSRP (Precio de Venta Sugerido por el Fabricante)- Reemplazar nulos por 0
if 'MSRP' in df_clean.columns:
    cantidad_nulos = df_clean['MSRP'].isnull().sum()
    if cantidad_nulos > 0:
        df_clean['MSRP'] = df_clean['MSRP'].fillna(0)
        print(f"      ✅ MSRP: {cantidad_nulos} nulos → 0")

# ==========================================================
# PASO 5: CONVERSIÓN DE TIPOS DE DATOS
# ==========================================================
print("\n📌 PASO 5: Conversión de tipos de datos")

# 5.1: Columnas numéricas
columnas_numericas = ['SALES', 'QUANTITYORDERED', 'PRICEEACH', 'MSRP']
for col in columnas_numericas:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        print(f"   ✅ {col} → numérico")

# 5.2: Fechas
if 'ORDERDATE' in df_clean.columns:
    df_clean['ORDERDATE'] = pd.to_datetime(df_clean['ORDERDATE'], errors='coerce')
    print(f"   ✅ ORDERDATE → datetime")

# 5.3: Columnas categóricas
columnas_categoricas = ['STATUS', 'PRODUCTLINE', 'COUNTRY', 'TERRITORY', 'DEALSIZE', 'CITY', 'STATE']
for col in columnas_categoricas:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype('category')
        print(f"   ✅ {col} → categorical")

# ==========================================================
# PASO 6: LIMPIEZA Y UNIFICACIÓN DE TEXTO (para columnas que no son categorical)
# ==========================================================
print("\n📌 PASO 6: Estandarización de texto")

# Columnas que deberían estar limpias (ya son categorical, pero verificamos)
for col in columnas_categoricas:
    if col in df_clean.columns:
        # Para categorical, convertimos a string, limpiamos, y volvemos a category
        valores_limpios = (
            df_clean[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.replace('NAN', 'DESCONOCIDO')
            .str.replace('DESCONOCIDO', 'DESCONOCIDO')  # Unificar
        )
        df_clean[col] = valores_limpios.astype('category')
        print(f"   ✅ {col} estandarizada")

# ==========================================================
# PASO 7: FEATURE ENGINEERING (CREACIÓN DE COLUMNAS DERIVADAS)
# ==========================================================
print("\n📌 PASO 7: Creación de columnas derivadas")

# 7.1: Calcular VALOR BRUTO
if 'QUANTITYORDERED' in df_clean.columns and 'PRICEEACH' in df_clean.columns:
    df_clean['GROSS_VALUE'] = df_clean['QUANTITYORDERED'] * df_clean['PRICEEACH']
    df_clean['GROSS_VALUE'] = df_clean['GROSS_VALUE'].round(2)
    print(f"   ✅ GROSS_VALUE: Cantidad × Precio unitario")

# 7.2: Calcular DESCUENTO APLICADO
if 'MSRP' in df_clean.columns and 'PRICEEACH' in df_clean.columns:
    df_clean['DISCOUNT_APPLIED'] = np.where(
        df_clean['MSRP'] > 0,
        (df_clean['MSRP'] - df_clean['PRICEEACH']) / df_clean['MSRP'],
        0
    )
    df_clean['DISCOUNT_APPLIED'] = df_clean['DISCOUNT_APPLIED'].round(4)
    print(f"   ✅ DISCOUNT_APPLIED: (MSRP - PRICE)/MSRP")

# 7.3: Extraer componentes de fecha
if 'ORDERDATE' in df_clean.columns:
    df_clean['YEAR'] = df_clean['ORDERDATE'].dt.year
    df_clean['MONTH'] = df_clean['ORDERDATE'].dt.month
    df_clean['QUARTER'] = df_clean['ORDERDATE'].dt.quarter
    df_clean['DAY_OF_WEEK'] = df_clean['ORDERDATE'].dt.dayofweek
    print(f"   ✅ Año, Mes, Trimestre y Día de semana extraídos")

# ==========================================================
# PASO 8: REDONDEO DE DECIMALES
# ==========================================================
print("\n📌 PASO 8: Redondeo de decimales")
columnas_redondear = ['SALES', 'PRICEEACH', 'MSRP', 'GROSS_VALUE']
for col in columnas_redondear:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].round(2)
        print(f"   ✅ {col}: redondeado a 2 decimales")

# ==========================================================
# PASO 9: DETECCIÓN DE VALORES ATÍPICOS (OUTLIERS)
# ==========================================================
print("\n📌 PASO 9: Detección de valores atípicos (outliers)")

def detectar_outliers(df, columna):
    if columna not in df.columns:
        return 0, None, None

    datos = df[columna].dropna()
    if len(datos) == 0:
        return 0, None, None

    Q1 = datos.quantile(0.25)
    Q3 = datos.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = datos[(datos < limite_inferior) | (datos > limite_superior)]

    return len(outliers), limite_inferior, limite_superior

for col in ['SALES', 'QUANTITYORDERED', 'PRICEEACH']:
    if col in df_clean.columns:
        cantidad, lim_inf, lim_sup = detectar_outliers(df_clean, col)
        if cantidad > 0:
            print(f"   ⚠️ {col}: {cantidad} outliers detectados")
            print(f"      Rango normal: {lim_inf:.2f} - {lim_sup:.2f}")
        else:
            print(f"   ✅ {col}: sin outliers")

# ==========================================================
# PASO 10: VERIFICACIÓN FINAL
# ==========================================================
print("\n" + "=" * 70)
print(" REPORTE FINAL DE LIMPIEZA")
print("=" * 70)

print(f"\n Dimensiones:")
print(f"   • Filas iniciales: {filas_iniciales}")
print(f"   • Filas finales: {len(df_clean)}")
print(f"   • Filas eliminadas: {filas_iniciales - len(df_clean)}")
print(f"   • Columnas finales: {len(df_clean.columns)}")

print(f"\n Calidad de datos:")
print(f"   • Duplicados restantes: {df_clean.duplicated().sum()}")
print(f"   • Nulos restantes: {df_clean.isnull().sum().sum()}")

if df_clean.isnull().sum().sum() == 0:
    print("   ✅ Dataset COMPLETAMENTE LIMPIO (sin nulos)")
else:
    print("   ⚠️ Aún hay nulos:")
    nulos_finales = df_clean.isnull().sum()
    print(nulos_finales[nulos_finales > 0])

print(f"\n Tipos de datos finales:")
for col, dtype in df_clean.dtypes.items():
    print(f"   • {col}: {dtype}")

print(f"\n Vista previa (primeras 5 filas):")
# Mostrar columnas más relevantes para no saturar
columnas_mostrar = ['ORDERNUMBER', 'SALES', 'QUANTITYORDERED', 'PRICEEACH',
                    'PRODUCTLINE', 'COUNTRY', 'ORDERDATE', 'GROSS_VALUE', 'DISCOUNT_APPLIED']
columnas_existentes = [col for col in columnas_mostrar if col in df_clean.columns]
print(df_clean[columnas_existentes].head(10))

# ==========================================================
# PASO 11: EXPORTAR
# ==========================================================
print("\n Exportando dataset limpio...")
df_clean.to_csv("sales_data_clean.csv", index=False)
print("   ✅ Archivo guardado: sales_data_clean.csv")

# Guardar reporte
with open("limpieza_report.txt", "w") as f:
    f.write("=== REPORTE DE LIMPIEZA DE DATOS ===\n\n")
    f.write(f"Filas iniciales: {filas_iniciales}\n")
    f.write(f"Filas finales: {len(df_clean)}\n")
    f.write(f"Columnas finales: {len(df_clean.columns)}\n")
    f.write(f"Columnas eliminadas: {columnas_a_eliminar_existentes}\n")
    f.write(f"Columnas creadas: GROSS_VALUE, DISCOUNT_APPLIED, YEAR, MONTH, QUARTER, DAY_OF_WEEK\n")
    f.write(f"Nulos restantes: {df_clean.isnull().sum().sum()}\n")
print("   ✅ Reporte guardado: limpieza_report.txt")

print("\n" + "=" * 70)
print(" LIMPIEZA COMPLETADA CON ÉXITO")
print("=" * 70)

INICIO DEL PROCESO DE NORMALIZACIÓN Y LIMPIEZA

📌 PASO 1: Inspección inicial
   Filas: 2823
   Columnas: 25

📌 PASO 2: Estandarización de nombres de columnas
   ✅ Columnas normalizadas

📌 PASO 2.5: Eliminación de columnas inútiles
   ✅ Eliminadas: ADDRESSLINE1, ADDRESSLINE2, PHONE, CONTACTLASTNAME, CONTACTFIRSTNAME, POSTALCODE

📌 PASO 3: Eliminación de filas duplicadas
   ✅ Duplicados eliminados: 0 → 0

📌 PASO 4: Tratamiento de valores nulos

   🔍 Análisis inicial de nulos:
      • STATE: 1486 nulos (52.6%)
      • TERRITORY: 1074 nulos (38.0%)

   🔧 Tratamiento de nulos:
      ✅ Eliminadas por datos críticos faltantes: 0
      ✅ TERRITORY: 1074 nulos → 'DESCONOCIDO'
      ✅ STATE: 1486 nulos → 'DESCONOCIDO'

📌 PASO 5: Conversión de tipos de datos
   ✅ SALES → numérico
   ✅ QUANTITYORDERED → numérico
   ✅ PRICEEACH → numérico
   ✅ MSRP → numérico
   ✅ ORDERDATE → datetime
   ✅ STATUS → categorical
   ✅ PRODUCTLINE → categorical
   ✅ COUNTRY → categorical
   ✅ TERRITORY → categorical
  